# EO Lexical Analysis — Experiment Notebook

## Cell 0: Setup & Initialization

This notebook runs **entirely in your browser** via [Pyodide](https://pyodide.org) (Python compiled to WebAssembly).
No server needed — everything executes client-side.

### What this cell does:
1. Installs required packages (numpy, scipy, scikit-learn)
2. Loads the analysis modules
3. Loads pre-computed data from the repository
4. Sets up API key configuration

**Run this cell first before any other notebook.**

In [ ]:
# ═══ Install packages (runs once, cached after) ═══
import micropip
await micropip.install(['numpy', 'scipy', 'scikit-learn'])
print('Packages installed: numpy, scipy, scikit-learn')

In [ ]:
# ═══ Load analysis modules from the repository ═══
from pyodide.http import pyfetch
import os, json

# Create module directory
os.makedirs('/home/pyodide/eo', exist_ok=True)

# Download Python modules
BASE_URL = '../py'  # Relative to JupyterLite deployment
modules = ['operator_definitions.py', 'analysis.py', 'conllu_parser.py', 'embeddings.py']

for mod in modules:
    try:
        resp = await pyfetch(f'{BASE_URL}/{mod}')
        text = await resp.string()
        with open(f'/home/pyodide/eo/{mod}', 'w') as f:
            f.write(text)
        print(f'  Loaded {mod} ({len(text)} bytes)')
    except Exception as e:
        print(f'  Failed to load {mod}: {e}')
        print(f'  (Will try loading from inline definitions instead)')

import sys
sys.path.insert(0, '/home/pyodide/eo')

# Verify imports
try:
    from operator_definitions import OPERATORS, HELIX_ORDER, TRIADS, ROLES
    print(f'\nOperator definitions loaded: {len(OPERATORS)} operators')
    print(f'Helix order: {HELIX_ORDER}')
except ImportError:
    print('Could not import operator_definitions — will define inline below')

In [ ]:
# ═══ Fallback: Inline operator definitions ═══
# (In case the file fetch fails)

try:
    HELIX_ORDER
except NameError:
    HELIX_ORDER = ['NUL','DES','INS','SEG','CON','SYN','ALT','SUP','REC']
    TRIADS = {
        'existence':      {'operators': ['NUL','DES','INS']},
        'structure':      {'operators': ['SEG','CON','SYN']},
        'interpretation': {'operators': ['ALT','SUP','REC']},
    }
    ROLES = {
        'ground':  ['NUL','SEG','ALT'],
        'figure':  ['DES','CON','SUP'],
        'pattern': ['INS','SYN','REC'],
    }
    print('Using inline operator definitions')

OP_COLORS = {
    'NUL': '#d4915c', 'DES': '#e8b84a', 'INS': '#f0d060',
    'SEG': '#4a9ec8', 'CON': '#5bb8d4', 'SYN': '#6dd4c8',
    'ALT': '#a87ccf', 'SUP': '#cf7cb8', 'REC': '#e07ca0',
}
print('Color scheme loaded')

In [ ]:
# ═══ Load pre-computed data from repository ═══
DATA = {}

data_files = {
    'verbs': '../data/verbs.json',
    'operators': '../data/operators.json',
    'metrics': '../data/metrics.json',
    'boundaries': '../data/boundaries.json',
    'crossling': '../data/crossling.json',
    'confusion': '../data/confusion.json',
    'influence': '../data/influence.json',
    'combined_corpus': '../data/combined_corpus.json',
}

for key, url in data_files.items():
    try:
        resp = await pyfetch(url)
        DATA[key] = json.loads(await resp.string())
        size = len(json.dumps(DATA[key]))
        print(f'  {key}: loaded ({size:,} bytes)')
    except Exception as e:
        print(f'  {key}: not found ({e})')

print(f'\nLoaded {len(DATA)} data files')

In [ ]:
# ═══ API Key Configuration ═══
# Keys are stored in Python variables only — they stay in your browser session.
# They are NEVER sent anywhere except to the API providers (OpenAI/Anthropic).

# Uncomment and paste your keys:
OPENAI_API_KEY = ''    # 'sk-...'
ANTHROPIC_API_KEY = '' # 'sk-ant-...'

if OPENAI_API_KEY:
    print('OpenAI API key configured')
else:
    print('OpenAI API key not set — embedding generation will not work')
    print('Set OPENAI_API_KEY above to enable')

if ANTHROPIC_API_KEY:
    print('Anthropic API key configured')
else:
    print('Anthropic API key not set — LLM classification will not work')
    print('Set ANTHROPIC_API_KEY above to enable')

In [ ]:
# ═══ Helper function: browser-based API calls ═══
from pyodide.http import pyfetch
import asyncio

async def api_call(url, headers, body):
    """Make a POST request from the browser."""
    resp = await pyfetch(
        url,
        method='POST',
        headers=headers,
        body=json.dumps(body),
    )
    return json.loads(await resp.string())

async def openai_embeddings(texts, model='text-embedding-3-large'):
    """Generate embeddings via OpenAI API."""
    if not OPENAI_API_KEY:
        raise ValueError('Set OPENAI_API_KEY in the Setup notebook')
    result = await api_call(
        'https://api.openai.com/v1/embeddings',
        {'Content-Type': 'application/json', 'Authorization': f'Bearer {OPENAI_API_KEY}'},
        {'model': model, 'input': texts}
    )
    if 'error' in result:
        raise ValueError(result['error']['message'])
    return [item['embedding'] for item in sorted(result['data'], key=lambda x: x['index'])]

async def anthropic_classify(prompt, system=''):
    """Call Anthropic API for classification."""
    if not ANTHROPIC_API_KEY:
        raise ValueError('Set ANTHROPIC_API_KEY in the Setup notebook')
    result = await api_call(
        'https://api.anthropic.com/v1/messages',
        {
            'Content-Type': 'application/json',
            'x-api-key': ANTHROPIC_API_KEY,
            'anthropic-version': '2023-06-01',
            'anthropic-dangerous-direct-browser-access': 'true',
        },
        {
            'model': 'claude-sonnet-4-20250514',
            'max_tokens': 4096,
            'system': system,
            'messages': [{'role': 'user', 'content': prompt}],
        }
    )
    if 'error' in result:
        raise ValueError(result['error']['message'])
    return result['content'][0]['text']

print('API helper functions defined: openai_embeddings(), anthropic_classify()')
print('\n Setup complete. Proceed to notebook 01_Corpus.')